<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    10 · Redes Neuronales: one-hot encoding · backpropagation · entrenamiento por epocas
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 5 — Del dato curado al modelo predictivo</em>
  </p>
</div>

# Primera Red Neuronal
---
### En esta lección aprenderás:

- cómo funciona el one-hot encoding.
- cómo programar una red neuronal simple con NumPy.
- cómo funciona la retropropagación (backpropagation).
- cómo actualizar los pesos de la red.
- cómo entrenar la red durante múltiples épocas.

---

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

### Funciones de activación

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x)) # np.exp() = e^()

def softmax(x, axis=1):
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

### One-hot encoding

Queremos que las etiquetas con las que entrenamos la red sean un vector de ceros y unos, en lugar de un número entero. 
Por ejemplo, queremos que el dígito `3` sea representado como `[0,0,0,1,0,0,0,0,0,0]`.

Esto se llama **one-hot encoding**. 
La posición del `1` en el vector corresponde al dígito. 
Para el dígito `3` el `1` está en la cuarta posición (índice 3).

In [ ]:
def one_hot(labels):
    one_hot_matrix = np.zeros(
        [
            len(labels),
            len(set(labels))
        ]
    )
    for i,x in enumerate(labels):
        one_hot_matrix[i,x] = 1
    return one_hot_matrix

In [ ]:
# Escala las imágenes originales, que pueden tener valores entre 0 y 255, a imágenes con valores entre 0 y 1.
# Esto permite que la red neuronal aprenda de manera más eficiente.
def min_max(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))

### Cargar los datos

Hoy trabajaremos con el dataset MNIST [1]. Es probablemente el dataset más famoso en el campo del aprendizaje profundo. 
Contiene 70.000 imágenes en escala de grises de dígitos escritos a mano (0–9). 
Cada imagen tiene un tamaño de 28x28 píxeles.

El objetivo es entrenar una red neuronal que pueda reconocer los dígitos correctamente.

\[1\] LeCun, Y. (1998). The MNIST database of handwritten digits. http://yann.lecun.com/exdb/mnist/

In [ ]:
train_data = np.genfromtxt('https://uni-muenster.sciebo.de/s/HvLen05c6fIzmyX/download', delimiter=',', skip_header =False) 
# genfromtxt reads .txt files, setting delimiter ="," allows to read .csv (comma seperated values) files.
test_data=np.genfromtxt('https://uni-muenster.sciebo.de/s/xdpCHFmmCAfeuxh/download', delimiter=',', skip_header =False) 
# Here we load the test data.

# After we have imported the data, we divide the data into images and labels.
# We also convert labels from float to integer with .astype(int).
train_labels=train_data[:,0].astype(int) 
train_images = train_data[:,1:]

test_labels=test_data[:,0].astype(int)
test_images = test_data[:,1:]

del train_data, test_data # for more memory we delete original data

Ahora hemos leído el dataset de entrenamiento. Con `train_images.shape` puedes ver que la variable `train_images` es un array de 60.000 filas y 784 columnas. 
Cada fila corresponde a una imagen. Cada imagen tiene 784 píxeles (28x28=784). 
La variable `train_labels` contiene las etiquetas correspondientes, es decir, los dígitos correctos.

In [ ]:
train_images.shape

In [ ]:
train_targets=one_hot(train_labels)
test_targets = one_hot(test_labels)

train_targets[:5,:] # the labels of the first five images

Finalmente, debemos usar el escalador `min_max` para escalar los valores de los píxeles entre cero y uno.

In [ ]:
train_images = min_max(train_images)
test_images = min_max(test_images)

En la siguiente celda puedes ver una imagen de ejemplo. La función `.reshape([28,28])` cambiará la imagen de vuelta a su forma original de 28x28 píxeles.

In [ ]:
plt.imshow(train_images[0].reshape([28, 28]), cmap="gray")
print("Correct label: %s" % train_labels[0])

### El modelo

**Inicializar los pesos**

Primero se deben crear las matrices de pesos y vectores de sesgo (bias) con el tamaño correcto. 
Los pesos se inicializan aleatoriamente (distribución normal con media 0 y desviación estándar 0.1). 
Los sesgos se inicializan con ceros.

La función `init_weights` toma tres argumentos:
- `input_size`: número de neuronas de entrada (784 para imágenes 28x28)
- `hidden_size`: número de neuronas en la capa oculta
- `output_size`: número de neuronas de salida (10 para los dígitos 0–9)

La función debe devolver dos listas: `W` con las matrices de pesos y `b` con los vectores de sesgo.

**¿Puedes completar la función?**

In [ ]:
# function initializing the weights
def init_weights(input_size, hidden_size, output_size):
    # two empty lists with a lenght of two
    b = [0] * 2
    W = [0] * 2

    # here the weights W are initialized with small, random numbers
    
    W[0] = np.random.randn(        ,         ) * np.sqrt(2 / input_size)  # WRITE THE CORRECT SIZES HERE
    W[1] = np.random.randn(        ,         ) * np.sqrt(2 / hidden_size) # WRITE THE CORRECT SIZES HERE

    # the bias can be zero
    b[0] = np.zeros(       ) # WRITE THE CORRECT SIZES HERE
    b[1] = np.zeros(       ) # WRITE THE CORRECT SIZES HERE


    return W, b

<details>
<summary><strong>Solución:</strong></summary>

```python
def init_weights(input_size, hidden_size, output_size):
    W = []
    b = []
    W.append(np.random.normal(0, 0.1, size=(hidden_size, input_size)))
    W.append(np.random.normal(0, 0.1, size=(output_size, hidden_size)))
    b.append(np.zeros(hidden_size))
    b.append(np.zeros(output_size))
    return W, b
```
</details>

Los pesos ya pueden inicializarse:

El tamaño de entrada está predefinido, ya que cada imagen tiene 784 píxeles. 
El tamaño de salida también está predefinido: tenemos 10 dígitos (0–9). 
Solo el tamaño de la capa oculta puede elegirse libremente.

In [ ]:
W, b = init_weights(input_size=784, hidden_size=200,output_size= 10)

In [ ]:
W[0].shape

Si inicializaste los pesos correctamente, `W[0].shape` debería ser `(200,784)`.

---

Puede que te preguntes: ¿por qué no simplemente inicializar todos los pesos con cero?

La respuesta es la **simetría**: si todos los pesos son iguales al inicio, todos los gradientes también serán iguales durante la retropropagación. 
Esto significa que todas las neuronas aprenderán exactamente lo mismo y la red nunca podrá aprender representaciones complejas.

**Forward pass**

Después de inicializar los pesos, se puede realizar el forward pass de la red. 
Las imágenes se pasan a través de la red y se obtiene una predicción.

El forward pass consiste en dos transformaciones lineales y dos funciones de activación:

$$Z_1 = XW_1^T + b_1$$
$$A_1 = \text{sigmoid}(Z_1)$$
$$Z_2 = A_1 W_2^T + b_2$$
$$\hat{Y} = \text{softmax}(Z_2)$$

**¿Puedes completar la función `forward_pass`?**

In [ ]:
def forward_pass(W, b, X):
    
    Z_1 = # CODE TO CALCULATE Z_1 
    A_1 = # CODE TO CALCULATE A_1 
    Z_2 = # CODE TO CALCULATE Z_2 
    Y_hat = #CODE TO CALCULATE Y_HAT 
    return Z_1, A_1, Y_hat

<details>
<summary><strong>Solución:</strong></summary>

```python
def forward_pass(W, b, X):
    Z_1 = np.matmul(X, W[0].T) + b[0]
    A_1 = sigmoid(Z_1)
    Z_2 = np.matmul(A_1, W[1].T) + b[1]
    Y_hat = softmax(Z_2)
    return Z_1, A_1, Y_hat
```
</details>

### Función de pérdida

Después del forward pass, se calcula la pérdida (loss). 
Mide qué tan bien o mal el modelo fue capaz de predecir las etiquetas correctas. 
En este caso usamos la **entropía cruzada categórica** (categorical cross-entropy):

$$L = -\sum_{i} y_i \log(\hat{y}_i)$$

Cuanto menor sea la pérdida, mejor será el modelo.

In [ ]:
def calc_loss(y_hat, y):
    return -np.sum(np.log(y_hat) * y)

Ahora puedes usar las tres primeras funciones juntas para realizar tu primera clasificación. 
Recuerda que las imágenes de `train_images` son las imágenes de entrenamiento y `train_targets` son las etiquetas en formato one-hot.

**¿Puedes completar el código?**

In [ ]:
np.random.seed(1234) # A seed is set so that the results of the random initialization are the same for all participants.
W, b = # CODE TO INITIALIZE THE WEIGHTS

# Use the train_images as X (input)
Z_1, A_1, Y_hat = # CODE FOR THE FORWARD PASS

# Here you calculate the loss
calc_loss(           ,           )/Y_hat.shape[0]

<details>
<summary><strong>Solución:</strong></summary>

```python
np.random.seed(1234)
W, b = init_weights(784, 300, 10)
Z_1, A_1, Y_hat = forward_pass(W, b, train_images)
print('Pérdida:', calc_loss(Y_hat, train_targets))
```
</details>

Adicionalmente calculamos la exactitud (accuracy) para tener una mejor idea de qué tan bien funciona nuestro modelo.

In [ ]:
def accuracy(true_labels,predicted):
    pred_labels = np.argmax(predicted, axis=1) # argmax returns the index of the maximum value
    correct_predicted = np.sum(true_labels == pred_labels)
    return correct_predicted /true_labels.shape[0]

In [ ]:
accuracy(train_labels,Y_hat)

Actualmente, la red tiene una exactitud del 10.3%. Una exactitud del 10% es lo que se espera si la red decide aleatoriamente. 
Para mejorar la exactitud, debemos ajustar los pesos. Esto se hace con la **retropropagación** (backpropagation).

**¿Puedes completar la función `back_prop`?**

In [ ]:
def back_prop(X, Z_1, A_1, Y_hat, y):
    n = X.shape[0] # n is the number of images
    
    # Gradients for the weights of the second layer
    dZ_2 =  # CODE TO CALCULATE dZ_2
    dW_2 =  # CODE TO CALCULATE dW_2
    db_2 = np.sum(dZ_2, axis=0) / n
    
    # Gradients for the weights of the first layer
    dZ_1 = np.multiply(np.matmul(dZ_2, W[1]), np.multiply(A_1, 1 - A_1))
    dW_1 = # CODE TO CALCULATE dW_1
    db_1 = # CODE TO CALCULATE db_1

    return [dW_1, dW_2], [db_1, db_2] # Here again two lists are returned, in each of the lists are the gradients for W_1,W_2 and b_1 and b_2.

<details>
<summary><strong>Solución:</strong></summary>

```python
def back_prop(X, Z_1, A_1, Y_hat, y):
    n = X.shape[0]
    dZ_2 = Y_hat - y
    dW_2 = np.matmul(dZ_2.T, A_1) / n
    db_2 = np.sum(dZ_2, axis=0) / n
    dA_1 = np.matmul(dZ_2, W[1])
    dZ_1 = dA_1 * A_1 * (1 - A_1)
    dW_1 = np.matmul(dZ_1.T, X) / n
    db_1 = np.sum(dZ_1, axis=0) / n
    return [dW_1, dW_2], [db_1, db_2]
```
</details>

### Actualizar los pesos

En el último paso, los pesos se ajustan. Para ello, los pesos se desplazan un poco en la dirección negativa del gradiente. 
Esto se llama **descenso de gradiente** (gradient descent).

$$W = W - \alpha \cdot \nabla W$$

Donde $\alpha$ es la **tasa de aprendizaje** (learning rate). 
Es un hiperparámetro que controla qué tan grandes son los pasos en la dirección del gradiente. 
Una tasa demasiado grande puede hacer que el modelo diverja; una demasiado pequeña puede hacer que el entrenamiento sea muy lento.

**¿Puedes completar la función `update`?**

In [ ]:
def update(W, b, grad_W, grad_b, lr=0.0001):
    W[0] = # CODE FOR W[0]
    W[1] = # CODE FOR W[1]
    b[0] = # CODE FOR b[0]
    b[1] = # CODE FOR b[1]

    return W, b # the function returns the new weights and biases (2 listn)

<details>
<summary><strong>Solución:</strong></summary>

```python
def update(W, b, grad_W, grad_b, lr=0.0001):
    W[0] = W[0] - lr * grad_W[0]
    W[1] = W[1] - lr * grad_W[1]
    b[0] = b[0] - lr * grad_b[0]
    b[1] = b[1] - lr * grad_b[1]
    return W, b
```
</details>

## Juntando todo

Ahora puedes juntar todo. Primero inicializas los pesos, luego la entrada se pasa a través de la red (forward pass), 
se calcula la pérdida y finalmente se ajustan los pesos (backpropagation + update).

**¿Puedes completar el código?**

In [ ]:
# 1.INITIALIZE WEIGHTS
np.random.seed(1234) 
W, b = init_weights(input_size=784, hidden_size=300,output_size= 10) 

# 2.FORWARD PROPAGATION
Z_1, A_1, Y_hat = forward_pass(W,b,train_images)

# 3.CALCULATE LOSS
print("Loss after first pass:",calc_loss(Y_hat,train_targets)/Y_hat.shape[0], "\nAccuracy after first pass:", accuracy(train_labels, Y_hat) )

# 4. BACKPROPAGATION
grad_W, grad_b = # CODE FOR BACKPROPAGATION

# 5. UPDATE WEIGHTS
W, b = # NEW WEIGHTS

<details>
<summary><strong>Solución:</strong></summary>

```python
# 1. INICIALIZAR PESOS
np.random.seed(1234)
W, b = init_weights(input_size=784, hidden_size=300, output_size=10)

# 2. FORWARD PASS
Z_1, A_1, Y_hat = forward_pass(W, b, train_images)
print('Pérdida antes del update:', calc_loss(Y_hat, train_targets))

# 3. RETROPROPAGACIÓN
grad_W, grad_b = back_prop(train_images, Z_1, A_1, Y_hat, train_targets)

# 4. ACTUALIZAR PESOS
W, b = update(W, b, grad_W, grad_b, lr=0.0001)
```
</details>

La pérdida y la exactitud aún no han cambiado. Solo cuando vuelves a pasar la entrada por la red puedes ver el efecto de la actualización de los pesos.

In [ ]:
Z_1, A_1, Y_hat = forward_pass(W,b,train_images)

# now calculate the loss
print("Loss after the second pass:",calc_loss(Y_hat,train_targets)/Y_hat.shape[0], "\nAccuracy after the second pass:", accuracy(train_labels, Y_hat) )

De hecho, la pérdida disminuye y la exactitud mejora. Sin embargo, también pueden ocurrir deterioros a corto plazo durante el entrenamiento, 
ya que el proceso de aprendizaje no siempre es monótono.

Para ver una mejora significativa, el proceso de entrenamiento debe repetirse muchas veces. 
Esto se puede lograr con un `for-loop`.

In [ ]:
# 1.INITALIZE WEIGHTS 
np.random.seed(1234)
W, b = init_weights(784, 300, 10)

EPOCHS= 50 # how often the data is passed through the network
for i in range(EPOCHS):
    
    # 2. FORWARD PROPAGATON
    Z_1, A_1, Y_hat = forward_pass(W, b,train_images)
    
    # 3. CALCULATE LOSS
    loss = calc_loss(Y_hat, train_targets) / Y_hat.shape[0]
    acc = accuracy(train_labels, Y_hat)
    
    print(i, 
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss, acc)
    )
    
    # 4. BACK PROPAGATION
    grad_W, grad_b = back_prop(train_images, Z_1, A_1, Y_hat, train_targets)
    
    # 5. UPDATE WEIGHTS
    W, b = update(W, b, grad_W, grad_b, lr = 0.1)

Ya puedes ver una mejora y alcanzar una exactitud del 73%. Sin embargo, el entrenamiento toma mucho tiempo. Con una tasa de aprendizaje más alta, el entrenamiento puede acelerarse.

In [ ]:
np.random.seed(1234)
W, b = init_weights(784, 300, 10)
loss= []
EPOCHS= 50 # how often the data is passed through the network
for i in range(EPOCHS):
    Z_1, A_1, Y_hat = forward_pass(W, b,train_images)
    
    loss= calc_loss(Y_hat, train_targets) / Y_hat.shape[0]
    
    acc = accuracy(train_labels, Y_hat)
    
    print(i,
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss, acc)
    )
    
    # BACK PROPAGATION
    grad_W, grad_b = back_prop(train_images, Z_1, A_1, Y_hat, train_targets)
    
    # UPDATE WEIGHTS
    W, b = update(W, b, grad_W, grad_b, lr = 0.3 )

Con una tasa de aprendizaje de 0.3, alcanzas una exactitud del 78% después de 50 épocas. Puedes mejorar aún más la red entrenándola por más épocas.

Sin embargo, no es necesario reinicializar los pesos. Simplemente pasa los datos por la red 25 épocas más.

**¿Puedes completar el código?**

In [ ]:
np.random.seed(1234)
W, b = init_weights(784, 300, 10)
loss= []
EPOCHS= 25 # how often the data is passed through the network
for i in range(EPOCHS):
    Z_1, A_1, Y_hat = forward_pass(W, b,train_images)
    loss= calc_loss(Y_hat, train_targets) / Y_hat.shape[0]
    acc = accuracy(train_labels, Y_hat)
    print(
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss, acc)
    )
    
    # BACK PROPAGATION
    grad_W, grad_b = back_prop(train_images, Z_1, A_1, Y_hat, train_targets)
    # UPDATE WEIGHTS
    W, b = update(W, b, grad_W, grad_b, lr = 0.3)

<details>
<summary><strong>Solución:</strong></summary>

```python
np.random.seed(1234)
# W, b = init_weights(784, 300, 10)  ← ¡NO reinicializar!
loss = []
EPOCHS = 25
for i in range(EPOCHS):
    Z_1, A_1, Y_hat = forward_pass(W, b, train_images)
    loss.append(calc_loss(Y_hat, train_targets))
    grad_W, grad_b = back_prop(train_images, Z_1, A_1, Y_hat, train_targets)
    W, b = update(W, b, grad_W, grad_b, lr=0.3)
plt.plot(loss)
print('Exactitud:', accuracy(train_labels, Y_hat))
```
</details>

Simplemente no debes reinicializar los pesos. De lo contrario, todo lo que la red ya ha aprendido se perderá.

Con 50 épocas adicionales (75 en total), ahora tienes una exactitud del 83%.

In [ ]:
_, _, test_y_hat = forward_pass(W, b, test_images) # by using underscores, z_1 and a_1 aren't returned, we don't need them
accuracy(test_labels, test_y_hat)   

La exactitud para el dataset de prueba también es del 85%. Es decir, el 85% de las imágenes fueron reconocidas correctamente. 
Es inusual que las redes neuronales tengan un mejor rendimiento en el conjunto de prueba que en el de entrenamiento. 
Esto puede ocurrir cuando el conjunto de entrenamiento es más difícil que el de prueba, o simplemente por aleatoriedad.

In [ ]:
false_classification = np.where(test_labels != np.argmax(test_y_hat, axis=1))[0]# which images were falsly classified
len(false_classification) # this many images were falsly classified

In [ ]:
# Model-probabilities that the image was correctly classified
probs = [] 
for image in false_classification:
    probs.append(test_y_hat[image,test_labels[image]])

In [ ]:
# We sort the images based on the probabilities, the smaller the probability 
# the more certain the model was that the image is not in the right category.
false_classification=false_classification[np.argsort(probs)]

In [ ]:
# this is what 10 images look like that are misclassified
for i in range(10):
    plt.imshow(test_images[false_classification[i]].reshape([28, 28]), cmap="gray")
    plt.show()
    print(
        "Predicted Label: %s, Correct Label %s"
        % (
            np.argmax(test_y_hat, axis=1)[false_classification[i]],
            test_labels[false_classification[i]],
        )
    )

Con algunas imágenes, puedes ver claramente por qué fueron clasificadas incorrectamente. 
Con otras, sin embargo, es bastante fácil para un humano ver qué dígito se muestra — aunque la red no pudo identificarlo correctamente.

Esto se debe a que la red aún no es perfecta. Para mejorarla, necesitaríamos más datos, más épocas o una arquitectura más compleja.

En el siguiente notebook aprenderás a usar **PyTorch**, una librería de deep learning que simplifica enormemente la implementación de redes neuronales.

In [ ]:
np.random.seed(1234)
W, b = init_weights(784, 300, 10)
loss= []
EPOCHS= 25 # how often the data is passed through the network
for i in range(EPOCHS):
    z_1, a_1, y_hat = forward_pass(W, b,train_images)
    loss= calc_loss(y_hat, train_targets) / y_hat.shape[0]
    acc = accuracy(train_labels, y_hat)
    print(
        "Training Loss: %.2f Training Accuracy: %.2f"
        % (loss, acc)
    )
    
    # backpropagation
    grad_W, grad_b = back_prop(train_images, z_1, a_1, y_hat, train_targets)
    # with the gradients we update the weights
    W, b = update(W, b, grad_W, grad_b, lr = 0.3)